# 第三部分：数据清洗（02_clean.ipynb 补充版）

这个 notebook 对应作业第三部分，主要完成：

- **3.1 单表清洗**：对每只股票原始日度数据进行缺失值检测与处理、日期统一、类型检查、重复值处理、异常涨跌幅标记
- **3.2 宽表与长表转换**：将 10 只股票收盘价拼成宽表，再转换回长表
- **3.3 多表合并**：将股票日度数据与指数日度数据、宏观月度数据合并，并记录每一步的行数变化

> 说明：本 notebook 默认你已经完成第一、二部分，因此项目根目录下已经存在  
> `data/stock/`、`data/index/`、`data/macro/`、`data/clean/`、`data/combined/` 等文件夹。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 200)

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start] + list(start.parents):
        if (p / "data" / "stock").exists():
            return p
    raise FileNotFoundError("未找到项目根目录。请把本 notebook 放在 dshw-p01 根目录后再运行。")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
STOCK_DIR = DATA_DIR / "stock"
INDEX_DIR = DATA_DIR / "index"
MACRO_DIR = DATA_DIR / "macro"
CLEAN_DIR = DATA_DIR / "clean"
COMBINED_DIR = DATA_DIR / "combined"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
COMBINED_DIR.mkdir(parents=True, exist_ok=True)

print("项目根目录：", PROJECT_ROOT)
print("股票目录：", STOCK_DIR)
print("指数目录：", INDEX_DIR)
print("宏观目录：", MACRO_DIR)

## 3.1 单表清洗（对每只股票原始数据）

这一部分的目标，是把每只股票的原始日度数据整理成**结构统一、类型正确、可直接进入分析**的数据表。  
清洗逻辑如下：

1. **缺失值检测**：先统计每列缺失值数量与比例，并解释可能原因  
2. **缺失值处理**：对数值列先进行 `ffill`，如果仍有无法处理的起始缺失，再删除对应记录  
3. **日期格式统一**：将 `date` 转为 `datetime64[ns]`，并在单表处理中设为索引  
4. **数据类型检查**：确保价格、成交量、成交额列都为数值类型  
5. **重复值处理**：按 `date + code` 检测重复并删除  
6. **离群值标记**：计算日收益率，若单日涨跌幅绝对值超过 20%，在 `is_extreme` 中标记为 `True`，但不删除

> 这里选择“**先向前填充，再删除剩余缺失**”的依据是：  
> 对日度金融时间序列而言，零星缺失通常意味着数据源在个别交易日未完整返回；先 `ffill` 可以保持时间序列连续性。  
> 如果缺失出现在序列最前端，前面没有可参考值，则只能删除这部分记录。

In [ ]:
# 读取 10 只股票的原始 CSV
stock_files = sorted([p for p in STOCK_DIR.glob("stock_*.csv") if p.name != "stock_universe.csv"])
if len(stock_files) == 0:
    raise FileNotFoundError("data/stock/ 下没有找到股票 CSV 文件。")

print("读取到的股票文件：")
for p in stock_files:
    print("-", p.name)

raw_stock_dict = {}
raw_overview = []

for fp in stock_files:
    df = pd.read_csv(fp)
    code_val = str(df["code"].iloc[0]).zfill(6) if ("code" in df.columns and len(df) > 0) else fp.stem.replace("stock_", "")
    raw_stock_dict[code_val] = df.copy()
    raw_overview.append({
        "code": code_val,
        "rows": len(df),
        "cols": len(df.columns),
        "columns": ", ".join(df.columns)
    })

raw_overview_df = pd.DataFrame(raw_overview).sort_values("code").reset_index(drop=True)
display(raw_overview_df)

### 3.1.1 缺失值检测

先对每只股票、每一列分别统计缺失值数量与缺失率。  
如果后续发现缺失主要集中在价格、成交量、成交额列，通常可能来自：

- 某些交易日数据源返回不完整
- 个别股票在某段时间存在停牌或异常状态
- 读取 CSV 时类型识别问题导致部分值被当成空值

如果缺失率接近 0，则说明原始下载结果整体质量较好，后续清洗压力较小。

In [ ]:
missing_records = []

for code_val, df in raw_stock_dict.items():
    n = len(df)
    for col in df.columns:
        miss_n = int(df[col].isna().sum())
        miss_ratio = miss_n / n if n > 0 else np.nan
        missing_records.append({
            "code": code_val,
            "column": col,
            "missing_count": miss_n,
            "missing_ratio": round(miss_ratio, 6)
        })

missing_summary = pd.DataFrame(missing_records).sort_values(["code", "column"]).reset_index(drop=True)
missing_summary.to_csv(CLEAN_DIR / "missing_value_summary.csv", index=False, encoding="utf-8-sig")

display(missing_summary.head(30))
print(f"缺失值统计已保存：{CLEAN_DIR / 'missing_value_summary.csv'}")

# 汇总到股票层面，便于写说明
missing_by_stock = (
    missing_summary.groupby("code", as_index=False)
    .agg(total_missing=("missing_count", "sum"),
         max_missing_ratio=("missing_ratio", "max"))
    .sort_values("code")
    .reset_index(drop=True)
)
display(missing_by_stock)

### 3.1.2 缺失值处理、日期统一、类型检查、重复值处理、离群值标记

这一部分真正执行清洗。清洗前后的变化会记录在汇总表中，包括：

- 原始行数 / 清洗后行数
- 删除的重复行数
- 处理前后缺失值总量
- 是否发生了类型转换
- 极端涨跌幅记录数

离群值的处理方式为“**标记，不删除**”。原因是：  
日收益率超过 ±20% 并不一定代表错误，它可能来自：

- 创业板、科创板等 20% 涨跌停制度
- 复权处理、除权除息带来的价格跳变
- 个别极端市场行情
- 数据源异常

因此，这类记录应先保留，并在后续分析中视研究目的决定是否进一步处理。

In [ ]:
numeric_cols = ["open", "close", "high", "low", "volume", "amount"]
cleaned_list = []
cleaning_summary_records = []

for code_val, raw_df in raw_stock_dict.items():
    df = raw_df.copy()
    before_rows = len(df)
    missing_before = int(df.isna().sum().sum())

    # 记录原始类型
    dtype_before = {col: str(df[col].dtype) for col in df.columns if col in df.columns}

    # 日期统一
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # 数值列统一转数值
    converted_cols = []
    for col in numeric_cols:
        if col in df.columns:
            old_dtype = str(df[col].dtype)
            df[col] = pd.to_numeric(df[col], errors="coerce")
            new_dtype = str(df[col].dtype)
            if old_dtype != new_dtype:
                converted_cols.append(col)

    # code 统一成 6 位字符串
    if "code" in df.columns:
        df["code"] = df["code"].astype(str).str.extract(r"(\d+)")[0].str.zfill(6)
    else:
        df["code"] = code_val

    # 重复值处理：按 date + code 判断
    dup_count = int(df.duplicated(subset=["date", "code"]).sum())
    df = df.drop_duplicates(subset=["date", "code"], keep="first").copy()

    # 排序后设索引
    df = df.sort_values(["date", "code"]).set_index("date")

    # 先对数值列做 ffill，再删除仍有关键字段缺失的记录
    cols_to_fill = [c for c in numeric_cols if c in df.columns]
    df[cols_to_fill] = df[cols_to_fill].ffill()

    # 删除 date 无法解析 或 关键数值仍缺失的行
    before_drop_na_rows = len(df)
    df = df[~df.index.isna()].copy()
    df = df.dropna(subset=[c for c in ["open", "close", "high", "low"] if c in df.columns]).copy()
    unresolved_na_drop = before_drop_na_rows - len(df)

    # 计算日收益率并标记极端值
    df["daily_return"] = df["close"].pct_change()
    df["is_extreme"] = df["daily_return"].abs() > 0.20
    extreme_count = int(df["is_extreme"].fillna(False).sum())

    # 恢复 date 为列，便于后续合并
    df = df.reset_index()

    after_rows = len(df)
    missing_after = int(df.isna().sum().sum())

    cleaned_list.append(df)

    cleaning_summary_records.append({
        "code": code_val,
        "rows_before": before_rows,
        "rows_after": after_rows,
        "duplicates_removed": dup_count,
        "rows_dropped_after_ffill": unresolved_na_drop,
        "missing_before": missing_before,
        "missing_after": missing_after,
        "converted_columns": ", ".join(converted_cols) if converted_cols else "",
        "extreme_count": extreme_count
    })

stock_clean = pd.concat(cleaned_list, ignore_index=True)
stock_clean = stock_clean.sort_values(["code", "date"]).reset_index(drop=True)

# 保存清洗结果
stock_clean.to_csv(CLEAN_DIR / "stock_clean.csv", index=False, encoding="utf-8-sig")

# 如环境支持，则顺手更新 parquet，便于与第二部分衔接
parquet_msg = "未写入 Parquet"
try:
    stock_clean.to_parquet(CLEAN_DIR / "stock_clean.parquet", index=False)
    parquet_msg = f"已写入 {CLEAN_DIR / 'stock_clean.parquet'}"
except Exception as e:
    parquet_msg = f"Parquet 写入跳过：{e}"

cleaning_summary = pd.DataFrame(cleaning_summary_records).sort_values("code").reset_index(drop=True)
cleaning_summary.to_csv(CLEAN_DIR / "cleaning_summary.csv", index=False, encoding="utf-8-sig")

display(cleaning_summary)
print(f"清洗后的股票面板已保存：{CLEAN_DIR / 'stock_clean.csv'}")
print(f"清洗汇总表已保存：{CLEAN_DIR / 'cleaning_summary.csv'}")
print(parquet_msg)

### 3.1.3 对清洗结果的文字说明

从清洗结果来看，通常有以下几种情况：

- **缺失值很少或几乎没有**：说明第一部分下载到的数据质量较好  
- **类型发生转换**：多见于 CSV 读入时把数值列识别成字符串，统一转成数值后更适合后续计算  
- **重复值数量为 0 或很少**：说明原始下载没有明显重复；即便为 0，也需要做这一步检查  
- **极端值数量通常不多**：这类观测不一定是错误，更可能代表制度性涨跌幅限制、复权跳点或极端行情

下面再额外查看被标记为 `is_extreme=True` 的记录，便于你在作业中写“可能原因”。

In [ ]:
extreme_obs = stock_clean.loc[stock_clean["is_extreme"] == True, 
                              ["date", "code", "close", "daily_return", "is_extreme"]].copy()
extreme_obs = extreme_obs.sort_values(["code", "date"]).reset_index(drop=True)

extreme_obs.to_csv(CLEAN_DIR / "extreme_observations.csv", index=False, encoding="utf-8-sig")
display(extreme_obs.head(30))
print(f"极端波动记录已保存：{CLEAN_DIR / 'extreme_observations.csv'}")

## 3.2 宽表与长表转换

这一部分只使用**收盘价**来演示结构变换。

### 思路说明

- **宽表（wide）**：日期作为索引，每一列是一只股票，更适合做相关系数、协方差矩阵、收益率比较、批量画图  
- **长表（long）**：每一行是一只股票某一天的一条观测，更适合做分组统计、面板数据建模、和其他表按键合并

因此，宽表和长表不是谁更“好”，而是适用于不同任务。  
下面先把 10 只股票的收盘价拼成宽表，再用 `pd.melt()` 转回长表。

In [ ]:
close_wide = (
    stock_clean[["date", "code", "close"]]
    .pivot_table(index="date", columns="code", values="close", aggfunc="first")
    .sort_index()
)

close_wide.to_csv(CLEAN_DIR / "stock_close_wide.csv", encoding="utf-8-sig")

close_long = (
    close_wide.reset_index()
    .melt(id_vars="date", var_name="code", value_name="close")
    .sort_values(["code", "date"])
    .reset_index(drop=True)
)

close_long.to_csv(CLEAN_DIR / "stock_close_long.csv", index=False, encoding="utf-8-sig")

print("宽表前 5 行：")
display(close_wide.head())

print("长表前 10 行：")
display(close_long.head(10))

print(f"宽表已保存：{CLEAN_DIR / 'stock_close_wide.csv'}")
print(f"长表已保存：{CLEAN_DIR / 'stock_close_long.csv'}")

### 3.2.1 文字回答：宽表与长表各适合什么操作？

- **宽表更适合**：
  - 计算多只股票之间的相关系数、协方差矩阵
  - 批量计算横截面收益率比较
  - 直接画多条时间序列图
  - 输入到某些要求“每列一个资产”的风险模型或优化模型

- **长表更适合**：
  - 使用 `groupby` 做按股票、按月份、按行业的统计
  - 与指数、宏观、财务等其他表按键合并
  - 做面板回归、固定效应模型
  - 保持 tidy data 结构，便于后续清洗和建模

## 3.3 多表合并

这一部分分两步：

1. **股票日度数据 + 指数日度数据**：按 `date` 做 `left join`  
2. **再与宏观月度数据合并**：先把月度宏观指标映射到对应月份的每个交易日，再合并到日度股票面板中

### 为什么要这样做？

- 股票和指数都是日频数据，按日期直接合并最自然  
- 宏观指标是月频，而股票是日频，频率不一致，不能直接按日期逐日匹配  
- 因此需要先把宏观数据转换成“按月键值”，再映射到每个交易日所属月份

In [ ]:
# ---------- 读取并整理指数数据 ----------
index_files = sorted(INDEX_DIR.glob("index_*.csv"))
if len(index_files) == 0:
    raise FileNotFoundError("data/index/ 下没有找到指数 CSV 文件。")

index_prepared = None
index_merge_log = []

for fp in index_files:
    idx = pd.read_csv(fp)
    idx["date"] = pd.to_datetime(idx["date"], errors="coerce")
    idx = idx.sort_values("date").drop_duplicates(subset=["date"]).copy()

    code_str = str(idx["code"].iloc[0]).zfill(6) if "code" in idx.columns and len(idx) > 0 else fp.stem.replace("index_", "")
    if code_str == "000300":
        prefix = "hs300"
    elif code_str == "000001":
        prefix = "sse"
    else:
        prefix = f"idx_{code_str}"

    idx["close"] = pd.to_numeric(idx["close"], errors="coerce")
    idx[f"{prefix}_ret"] = idx["close"].pct_change()

    keep_cols = ["date", "close", f"{prefix}_ret"]
    idx_small = idx[keep_cols].rename(columns={"close": f"{prefix}_close"})

    if index_prepared is None:
        index_prepared = idx_small.copy()
    else:
        before = len(index_prepared)
        index_prepared = index_prepared.merge(idx_small, on="date", how="outer")
        after = len(index_prepared)
        index_merge_log.append({
            "step": f"merge_{prefix}",
            "rows_before": before,
            "rows_after": after
        })

index_prepared = index_prepared.sort_values("date").reset_index(drop=True)

# ---------- 股票 + 指数 left join ----------
stock_daily = stock_clean.copy()
stock_daily["date"] = pd.to_datetime(stock_daily["date"], errors="coerce")

rows_before_index_merge = len(stock_daily)
stock_index_merged = stock_daily.merge(index_prepared, on="date", how="left")
rows_after_index_merge = len(stock_index_merged)

index_row_log = pd.DataFrame([{
    "step": "stock_left_join_index",
    "rows_before": rows_before_index_merge,
    "rows_after": rows_after_index_merge,
    "reason": "左连接保留了全部股票交易日观测；指数只是为每个日期补充市场信息，因此行数通常不变。"
}])

display(index_prepared.head())
display(index_row_log)

### 3.3.1 股票与指数合并后的变化说明

采用 `left join` 后，**以股票日度面板为主表**。  
这意味着：

- 股票表中的每一行都会保留
- 如果某天指数表有对应日期，就把指数列补进来
- 如果指数表没有对应日期，则相关列为缺失

因此，合并前后**行数通常保持不变**；变化主要体现在**列数增加**，因为加入了市场基准信息。

In [ ]:
# ---------- 读取并整理宏观月度数据 ----------
macro_files = sorted(MACRO_DIR.glob("macro_*.csv"))
if len(macro_files) == 0:
    raise FileNotFoundError("data/macro/ 下没有找到宏观 CSV 文件。")

macro_wide_parts = []

for fp in macro_files:
    m = pd.read_csv(fp)
    # 兼容字段：date / 日期, value / 今值
    date_col = "date" if "date" in m.columns else ("日期" if "日期" in m.columns else None)
    value_col = "value" if "value" in m.columns else ("今值" if "今值" in m.columns else None)

    if date_col is None or value_col is None:
        print(f"跳过 {fp.name}：未识别到 date/value 字段")
        continue

    m[date_col] = pd.to_datetime(m[date_col], errors="coerce")
    m[value_col] = pd.to_numeric(m[value_col], errors="coerce")

    if "indicator" in m.columns and m["indicator"].notna().any():
        indicator_name = str(m["indicator"].dropna().iloc[0])
    else:
        indicator_name = fp.stem

    if "cpi" in fp.stem.lower() or "cpi" in indicator_name.lower():
        col_name = "cpi_yoy"
    elif "lpr" in fp.stem.lower() or "lpr" in indicator_name.lower():
        col_name = "lpr_1y"
    else:
        col_name = re.sub(r"[^0-9a-zA-Z_]+", "_", fp.stem.lower())

    one = m[[date_col, value_col]].rename(columns={date_col: "macro_date", value_col: col_name}).copy()
    one["year_month"] = one["macro_date"].dt.to_period("M").astype(str)
    one = one.sort_values("macro_date").drop_duplicates(subset=["year_month"], keep="last")
    one = one[["year_month", col_name]]

    macro_wide_parts.append(one)

if len(macro_wide_parts) == 0:
    raise ValueError("宏观文件存在，但没有成功整理出可合并的数据。")

macro_wide = macro_wide_parts[0].copy()
for part in macro_wide_parts[1:]:
    macro_wide = macro_wide.merge(part, on="year_month", how="outer")

macro_wide = macro_wide.sort_values("year_month").reset_index(drop=True)
macro_wide.to_csv(CLEAN_DIR / "macro_monthly_wide.csv", index=False, encoding="utf-8-sig")

# ---------- 将宏观月度数据映射到日度股票面板 ----------
stock_index_merged["year_month"] = stock_index_merged["date"].dt.to_period("M").astype(str)

rows_before_macro_merge = len(stock_index_merged)
combined_data = stock_index_merged.merge(macro_wide, on="year_month", how="left")
rows_after_macro_merge = len(combined_data)

merge_row_log = pd.DataFrame([
    {
        "step": "stock_left_join_index",
        "rows_before": rows_before_index_merge,
        "rows_after": rows_after_index_merge,
        "reason": "左连接以股票交易日为主表，通常行数不变。"
    },
    {
        "step": "daily_left_join_macro_by_month",
        "rows_before": rows_before_macro_merge,
        "rows_after": rows_after_macro_merge,
        "reason": "月度宏观指标按所属月份映射到每个交易日，属于补充列信息，通常不改变行数。"
    }
])

combined_data = combined_data.sort_values(["code", "date"]).reset_index(drop=True)
combined_data.to_csv(COMBINED_DIR / "combined_data.csv", index=False, encoding="utf-8-sig")
merge_row_log.to_csv(CLEAN_DIR / "merge_row_log.csv", index=False, encoding="utf-8-sig")

display(macro_wide.head())
display(merge_row_log)
print(f"最终合并数据已保存：{COMBINED_DIR / 'combined_data.csv'}")
print(f"行数变化记录已保存：{CLEAN_DIR / 'merge_row_log.csv'}")

### 3.3.2 关于“频率不一致”的文字说明

股票和指数是**日频**数据，而 CPI、LPR 等宏观变量通常是**月频**数据。  
如果直接按完整日期去合并，会出现大量匹配不到的情况，因为一个月的宏观值只公布一次，但股票每天都在交易。

因此，本 notebook 采用的处理方法是：

1. 先把宏观数据转成 `year_month`
2. 再把每条股票日度观测映射到它所属的月份
3. 最后按 `year_month` 做左连接

这样做的优点是：

- 能把同一个月的宏观环境信息分配到该月所有交易日
- 保留股票日频数据的完整性
- 便于后续做按日回归、按月聚合或面板分析

同时也要注意，这种做法隐含了一个假设：  
**同一个月内，宏观变量对交易日的影响在信息层面近似相同。**

## 第三部分完成后的建议检查

运行完本 notebook 后，你可以重点检查这些文件：

- `data/clean/missing_value_summary.csv`
- `data/clean/cleaning_summary.csv`
- `data/clean/extreme_observations.csv`
- `data/clean/stock_clean.csv`
- `data/clean/stock_close_wide.csv`
- `data/clean/stock_close_long.csv`
- `data/clean/macro_monthly_wide.csv`
- `data/clean/merge_row_log.csv`
- `data/combined/combined_data.csv`

如果老师要求你在 `02_clean.ipynb` 中展示文字说明，那么本 notebook 里的 markdown 内容就可以直接满足“**不能只有代码**”这一要求。

In [ ]:
print("第三部分全部完成。你现在可以检查以下文件：")
for p in [
    CLEAN_DIR / "missing_value_summary.csv",
    CLEAN_DIR / "cleaning_summary.csv",
    CLEAN_DIR / "extreme_observations.csv",
    CLEAN_DIR / "stock_clean.csv",
    CLEAN_DIR / "stock_close_wide.csv",
    CLEAN_DIR / "stock_close_long.csv",
    CLEAN_DIR / "macro_monthly_wide.csv",
    CLEAN_DIR / "merge_row_log.csv",
    COMBINED_DIR / "combined_data.csv",
]:
    print("-", p)